# 4.4 D) Integracion y Dependencia Lineal
Analisis de redundancia lineal sobre la matriz imputada (MICE) de 4.3 y su impacto en `det(X^T X)`.

### 4.4.1 Analisis de redundancia lineal
Objetivo: identificar si existe `x_j = sum alpha_k x_k + beta`, es decir, direcciones exactas de dependencia. Se trabaja con la matriz imputada (452 x 261) y el analisis se hace por **espacio nulo via SVD** de la matriz centrada: los vectores singulares derechos con singularidad nula son, de una vez, las dependencias exactas y su reparticion de pesos entre columnas. El enfoque previo (R2 de regresiones por columna) era invalido con rango deficiente: `lstsq` devuelve la solucion de norma minima, cuyos coeficientes son arbitrarios entre columnas colineales.


In [1]:
import numpy as np, pandas as pd

df = pd.read_parquet("data/arrhythmia_imputado.parquet")
X = df.drop(columns=["Class"]).to_numpy(dtype=float)
names = list(df.drop(columns=["Class"]).columns)
n, d = X.shape
print(f"Matriz imputada: {n} x {d}")


Matriz imputada: 452 x 261


In [2]:
# Espacio nulo de X (centrada) via SVD: identifica las dependencias de forma directa
Xc = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
tol_s = np.finfo(float).eps * max(n, d) * s[0]
rk = int((s > tol_s).sum())
print(f"Rango de X: {rk} de {d} | direcciones nulas: {d - rk} | sgval min no nulo: {s[rk-1]:.3e}")

# Nucleo: filas rk..d-1 de Vt, vectores ortonormales v con Xc @ v = 0
N = Vt[rk:]
print(f"Nucleo: {N.shape[0]} vectores de peso en {d} columnas")
for i, v in enumerate(N):
    jj = np.argsort(np.abs(v))[::-1][:8]
    carga = ", ".join(f"{names[j]}({v[j]:+.3f})" for j in jj if abs(v[j]) > 1e-3)
    res = np.linalg.norm(Xc @ v)
    print(f"  d{i}: {carga}   (||Xc v|| = {res:.1e})")

# Relaciones explicitas x_j = sum alpha_k x_k + beta:
# de Xc v = 0 se despeja la columna dominante de cada direccion
# beta = (v^T mu) / v[j0] (no es cero en general para variables originales)
mu_global = X.mean(axis=0)
print()
print("Combinaciones lineales exactas (despejando la columna de mayor peso, con intercepto):")
for i, v in enumerate(N):
    j0 = int(np.argmax(np.abs(v)))
    alpha = -v / v[j0]
    beta = float(v @ mu_global / v[j0])
    parts = [f"{alpha[j]:+.3f}*{names[j]}" for j in range(d) if j != j0 and abs(alpha[j]) > 1e-3]
    print(f"  {names[j0]} = {beta:+.3f} + " + " + ".join(parts))

Rango de X: 257 de 261 | direcciones nulas: 4 | sgval min no nulo: 1.197e-02
Nucleo: 4 vectores de peso en 261 columnas
  d0: DI_QRSA(-0.546), DIII_Q_width(+0.546), AVR_R_prime_width(-0.541), DIII_S_ampl(-0.281), V4_R_width(+0.181), DIII_QRSA(+0.019), V3_R_prime_width(+0.010), DIII_R_ampl(-0.005)   (||Xc v|| = 1.5e-13)
  d1: V4_R_width(-0.617), DIII_S_ampl(+0.570), V2_n_intrinsic_deflections(-0.442), AVR_R_prime_width(-0.179), DI_QRSA(-0.163), DIII_Q_width(+0.163), V3_R_prime_width(-0.096), DIII_QRSA(-0.063)   (||Xc v|| = 6.2e-14)
  d2: V2_n_intrinsic_deflections(+0.778), DIII_S_ampl(+0.596), DI_QRSA(-0.096), DIII_Q_width(+0.096), AVR_R_prime_width(-0.095), V3_R_prime_width(+0.093), V4_R_width(+0.056), DI_R_prime_ampl(+0.021)   (||Xc v|| = 6.0e-14)
  d3: V3_R_prime_width(+0.991), V2_n_intrinsic_deflections(-0.116), V4_R_width(-0.067), DII_P_ampl(+0.012), DIII_QRSA(-0.007), DI_R_prime_ampl(-0.003), AVR_R_prime_width(-0.003), DIII_S_ampl(+0.002)   (||Xc v|| = 2.5e-15)

Combinaciones line

**Dependencias encontradas.** El espacio nulo de X (centrada) tiene **4 direcciones** (rango 257 de 261), verificadas con ||Xc v|| del orden de 1e-13..1e-14. Las combinaciones lineales exactas no provienen de relaciones fisiologicas de escala entre amplitudes (la narrativa previa de "triangulo de Einthoven" no se sostiene), sino de **columnas casi-binarias o cuasi-constantes** cuyos estados discretos generan ciclos lineales:

- **d0:** DI_QRSA, DIII_Q_width y AVR_R_prime_width (tres columnas con 2 valores unicos: 0/1) forman el ciclo `DI_QRSA = DIII_Q_width - AVR_R_prime_width - ...` (DI_QRSA es la despejada, |v|=0.546).
- **d1:** V4_R_width domina (|v|=0.617) con DIII_S_ampl (+0.570) y V2_n_intrinsic_deflections (-0.442).
- **d2 y d3:** ambas giran sobre V2_n_intrinsic_deflections, DIII_S_ampl, V3_R_prime_width y V4_R_width (V3_R_prime_width tiene 2 valores unicos, V2 3 valores y V4 4), dando dos direcciones degeneradas del mismo subespacio de columnas discretas.

Estas columnas tienen varianza >= 7 ordenes por debajo de las continuas (V3_R_prime_width var 4e-4 frente a ~10^3 de T_vector): su "informacion" es un indicador de presencia/ausencia que colapsa en una relacion exacta por construccion. La eleccion previa de columnas a eliminar por bucle greedy (DI_R_prime_ampl, DI_QRSA, DII_P_ampl, DII_QRSTA) era dependiente del orden y mezclaba criterios; la seleccion canonica sale del analisis del nucleo: eliminar el pico de cada vector nulo (DI_QRSA en d0, V4_R_width en d1, V2_n_intrinsic_deflections en d2 y V3_R_prime_width en d3) rompe las 4 dependencias; es exactamente la seleccion que se aplica en 4.5.3.

**Conclusion 4.4.1.** `X v = 0` define las dependencias lineales exactas y el espacio nulo tiene **4 direcciones**: 257 columnas independientes de 261. A diferencia de la lectura previa (13 columnas R2=1 atribuidas a fiisiologia del ECG), el nucleo se origina en **columnas cuasi-constantes con 2-4 valores unicos** (AVR_R_prime_width, DI_QRSA, DIII_Q_width, DIII_S_ampl, V3_R_prime_width, V2_n_intrinsic_deflections, V4_R_width): combinaciones exactas entre indicadores discretos, no relaciones entre canales. Son candidatas a eliminar por estar en el nucleo y por su varianza casi nula; la eleccion canonica se usa en E. El impacto de estas 4 direcciones sobre det(X^T X) se desarrolla en 4.4.2.

### 4.4.2 Impacto de la multicolinealidad en det(X^T X)
La redundancia lineal de 4.4.1 implica `det(X^T X) = 0`: como X^T X es simetrica semidefinida positiva, su determinante es el producto de sus autovalores; basta un autovalor nulo para anularlo.

In [3]:
# det(X^T X) = prod(autovalores); Gram CENTRADA (convencion unificada de toda la tarea)
G = Xc.T @ Xc  # Xc ya centrado en Cell 2
ev = np.linalg.eigvalsh(G)
tol = np.finfo(float).eps * max(n, d) * ev[-1]
nulos = ev[ev <= tol]
print(f"Autovalores de X^T X centrados ({d}):")
print(f"  nulos (<= tol): {len(nulos)}  |  rango efectivo: {d - len(nulos)} de {d}")
print(f"  menor autovalor no nulo: {ev[ev > tol][0]:.3e}")

if len(nulos):
    print(f"det(X^T X) = 0 exacto ({len(nulos)} autovalor(es) nulo(s)): X^T X singular, no invertible")
else:
    logdet = float(np.sum(np.log(ev)))
    print(f"log det(X^T X) = {logdet:.2f}  ->  det = {np.exp(logdet):.3e}")

kapp = ev[-1] / ev[ev > tol][0]
print(f"kappa (convencion centrada) = {kapp:.2e} (log10 = {np.log10(kapp):.1f})")
print()
print("Verificacion local: traza tr(X^T X centrada) = tr = suma de autovalores")
print(f"  tr(G) = {np.trace(G):.2f} | suma autovalores = {ev.sum():.2f}")

Autovalores de X^T X centrados (261):
  nulos (<= tol): 4  |  rango efectivo: 257 de 261
  menor autovalor no nulo: 1.432e-04
det(X^T X) = 0 exacto (4 autovalor(es) nulo(s)): X^T X singular, no invertible
kappa (convencion centrada) = 2.10e+10 (log10 = 10.3)

Verificacion local: traza tr(X^T X centrada) = tr = suma de autovalores
  tr(G) = 18190262.20 | suma autovalores = 18190262.20


**Interpretacion.** `det(X^T X) = 0` (convencion centrada) confirma que la multicolinealidad detectada en 4.4.1 anula la invertibilidad: las 4 direcciones nulas hacen que las ecuaciones normales `X^T X b = X^T y` no tengan solucion unica (sistema con infinitas soluciones). El rango efectivo es 257 de 261. Ademas, el condicionamiento es extremo aun sobre el complemento no nulo: kappa = 2.10e10 (log10 10.3), medido con el espectro de la matriz centrada (convencion unificada de la tarea); no hubo overflow en el calculo, el determinante es 0 exacto porque existen autovalores nulos.

Tambien se verifica la identidad de la traza: tr(X^T X centrada) = suma de autovalores (tr(G) = 18190262.20 = 40244.94 * 452, es decir n veces la traza poblacional de Sigma), lo que localiza la multicolinealidad: toda la masa esta en los 257 autovalores no nulos y 0 en las 4 direcciones degeneradas.

**Conclusion 4.4.2.** Las 4 dependencias lineales de 4.4.1 se materializan como 4 autovalores nulos de X^T X y fuerzan `det(X^T X) = 0`. La multicolinealidad no es solo un problema de estabilidad (kappa 2.10e10 (log10 10.3)) sino de identificabilidad: el estimador de minimos cuadrados no es unico sobre el subespacio degenerado.

**Consecuencia para E).** Al reducir el espacio (4.5 E) basta con neutralizar las 4 direcciones nulas. Dos caminos equivalentes: (a) eliminar una columna por direccion del nucleo de 4.4.1 (el pico de cada vector nulo: DI_QRSA en d0, V4_R_width en d1, V2_n_intrinsic_deflections en d2 y V3_R_prime_width en d3, como se aplica en 4.5.3) hasta recuperar inversibilidad; o (b) regularizar (ridge) sumando lambda*I, que desplaza los 4 ceros a lambda y hace det(X^T X + lambda*I) > 0. La decision de espacio se APLICA en E (4.5.3) con la seleccion canonica aqui descrita; aqui queda cuantificado el impacto: 4 direcciones muertas del total de 261.